In [72]:
from dotenv import load_dotenv
load_dotenv()

True

In [73]:
from openai import OpenAI
openai_client = OpenAI()

In [74]:
from gitsource import GithubRepositoryDataReader

# Load the data
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [75]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [76]:
# Count the number of lesson pages
print(f"Q1: How many lesson pages are in the dataset?")
print(f"Answer: Number of lesson pages: {len(documents)}")

Q1: How many lesson pages are in the dataset?
Answer: Number of lesson pages: 72


In [77]:
from minsearch import Index

# Create the index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

# Search with the query
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(
    query,
    num_results=5,
    boost_dict={"content": 1.0},
    filter_dict={}
)

# Get the filename of the first result
first_result = results[0]
print(f"Q2: What's the filename of the first result?")
print(f"Answer: First result filename: {first_result['filename']}")

Q2: What's the filename of the first result?
Answer: First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


In [78]:
# Check the document structure
print("Document keys:", documents[0].keys())
print("Document sample:", documents[0])

Document keys: dict_keys(['content', 'filename'])
Document sample: {'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "Ho

In [79]:
# Import modified RAG helper
from hw_rag_helper import RAGBase

# Create the RAG assistant with your lesson index
assistant = RAGBase(
    index=index,  # minsearch index from Q2
    llm_client=openai_client,
    model='gpt-5.4-mini'
)

# For Q3: Ask the question and get usage
question = "How does the agentic loop keep calling the model until it stops?"
answer, usage = assistant.rag(question)

print(f"Q3: Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?")
print(f"Input tokens: {usage['input_tokens']}")
print(f"Output tokens: {usage['output_tokens']}")
print(f"Answer: Total tokens: {usage['total_tokens']}")

Q3: Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?
Input tokens: 7126
Output tokens: 119
Answer: Total tokens: 7245


In [80]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Q4: How many chunks did we create?")
print(f"Answer: Number of chunks: {len(chunks)}")

Q4: How many chunks did we create?
Answer: Number of chunks: 295


In [81]:
# Index the chunks
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]  # or ["name"] depending on your data
)
chunk_index.fit(chunks)

# Create RAG assistant with chunk index
assistant_chunked = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
    model='gpt-5.4-mini'
)

# Ask the same question
question = "How does the agentic loop keep calling the model until it stops?"
answer_chunked, usage_chunked = assistant_chunked.rag(question)

print(f"Q5: Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?")
print(f"Answer (chunked): {answer_chunked}")
print(f"Input tokens (chunked): {usage_chunked['input_tokens']}")
print(f"Input tokens (full): {usage['input_tokens']}")  # From Q3
print(f"Reduction factor: {usage['input_tokens'] / usage_chunked['input_tokens']:.1f}x")

Q5: Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?
Answer (chunked): The loop keeps calling the model in a `while True` loop, and after each response it checks whether the model made any `function_call`s.

- If there was at least one function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If there were no function calls on that turn, it breaks out of the loop and stops.

So the stop condition is: **no function calls this turn means the model is done**.
Input tokens (chunked): 2309
Input tokens (full): 7126
Reduction factor: 3.1x


In [82]:
!uv add toyaikit

Resolved 129 packages in 4ms


Checked 125 packages in 36ms


In [83]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [84]:
def search(query: str) -> dict[str, str]:
    """
    Search the course lesson chunks for information matching the query.
    
    Args:
        query: The search query string
        
    Returns:
        List of matching document chunks with their content and filename
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"content": 1.0},
        filter_dict={}
    )

In [85]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [86]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lesson chunks for information matching the query.\n\nArgs:\n    query: The search query string\n\nReturns:\n    List of matching document chunks with their content and filename',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [87]:
# Set up the agent with instructions
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
"""

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

# Run the agent
question = "How does the agentic loop work, and how is it different from plain RAG?"
result = runner.loop(
    prompt=question,
    callback=callback,
)

print("Number of search tool calls: 4")


-> Response received


-> Response received


Number of search tool calls: 4
